# **Copyright and Author Information**

Copyright © [2026] Docketrun Tech Private Limited. All rights reserved.

This notebook is open-sourced and available under the [MIT License](https://opensource.org/licenses/MIT).

Author: Yahiya Mulla

This Colab notebook is provided for educational and informational purposes only. You are free to use, modify, and distribute it, provided that proper attribution is given.

**Disclaimer:** The information in this notebook is provided "as is," without warranty of any kind. The authors or the company do not accept any responsibility for errors or omissions in the content.

# How AI Sees: From Raw Pixels to Deep Understanding with VGG16

## Step 1: Download the Input Image

Before we can visualize the network's features, we need a sample image. The following function uses the requests library to fetch a high-quality image from the web and save it locally as cat.jpg. We include a 'User-Agent' header to mimic a real browser, preventing access errors from the server.

In [ ]:
import requests

def download_image(url, save_path="input.jpg"):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    with open(save_path, "wb") as f:
        f.write(response.content)

    print(f"Image downloaded and saved to {save_path}")

img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
download_image(img_url, "cat.jpg")

## Step 2: Load and Display the Input

Now that the image is saved locally, we load it using the PIL library and convert it to RGB format to ensure it has the standard 3 color channels. We also configure the PyTorch device (CPU or GPU) and display the image using Matplotlib to confirm it loaded correctly before processing.

In [ ]:
import torch
from PIL import Image
import matplotlib.pyplot as plt

# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_image_from_file(path):
    img = Image.open(path).convert("RGB")
    return img

# Load image
original_image = load_image_from_file("cat.jpg")

# Display image
plt.imshow(original_image)
plt.axis("off")
plt.title("Input Image")
plt.show()

## Step 3: Manual Convolution Demo (The "Magnifying Glass")

Before using the full VGG16 model, let's manually replicate the core mechanism of a CNN: the Convolution.

In this block, we convert the image to grayscale and apply a specific 3x3 Kernel (a vertical edge detector). By sliding this filter over the image, we perform a mathematical "dot product" at every pixel. This process highlights specific patterns, in this case, vertical lines like the cat's whiskers, demonstrating exactly how a neural network extracts features from raw pixels.

In [ ]:
import torch.nn as nn
from torchvision import models, transforms, utils
import numpy as np
import cv2

def manual_convolution_demo(image_pil):
    # Convert image to grayscale for simpler 2D convolution
    img_gray = np.array(image_pil.convert('L'))

    # Define a 3x3 Kernel (Vertical Edge Detector)
    # This matches the "Filter" concept in your Slide 1
    kernel = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ])

    # Apply the filter (The "Sliding Window")
    # This performs the dot product at every pixel
    convolved_image = cv2.filter2D(img_gray, -1, kernel)

    # Visualization
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    ax[0].imshow(img_gray, cmap='gray')
    ax[0].set_title("1. Input (Grayscale)")

    # Visualizing the Kernel (The Filter)
    ax[1].imshow(kernel, cmap='gray')
    ax[1].set_title("2. The Filter (Vertical Edge Kernel)")
    for (i, j), val in np.ndenumerate(kernel):
        ax[1].text(j, i, f'{val}', ha='center', va='center', color='red', fontsize=20)

    # Visualizing the Result (Feature Map)
    ax[2].imshow(convolved_image, cmap='gray')
    ax[2].set_title("3. The Feature Map (Result)")

    plt.show()
    print("Observation: Notice how vertical lines (like the cat's whiskers or ears) are bright, while horizontal areas are dark. The math 'found' the vertical edges.")

manual_convolution_demo(original_image)

## Step 4: The "X-Ray" Setup (Hooks & Model Loading)

Now we move from manual math to Deep Learning. We load a pre-trained VGG16 model, which has already "learned" to see millions of images. To see what happens inside its "brain" as it processes our cat image, we attach Hooks (listeners) to three specific layers:

- Low-Level (Layer 0): To see basic edges and colors.

- Mid-Level (Layer 14): To see shapes and textures.

- High-Level (Layer 28): To see complex object parts.

When we run the final line (output = model(input_tensor)), the image passes through the network, and our hooks automatically capture the hidden "Feature Maps" at each stage.

In [ ]:
# 1. Load Pre-trained VGG16
# We use VGG16 because it has a simple structure (Stacked Conv Layers) that maps perfectly to your diagram.
model = models.vgg16(pretrained=True).to(device)
model.eval() # Set to evaluation mode

# 2. Preprocess the image for the AI
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess(original_image).unsqueeze(0).to(device)

# 3. Define Hooks to capture the feature maps
feature_maps = {}

def get_activation(name):
    def hook(model, input, output):
        feature_maps[name] = output.detach()
    return hook

# Register hooks at different depths
# Layer 0: Very first convolution (Low-level features)
model.features[0].register_forward_hook(get_activation('layer_1_low_level'))
# Layer 14: Middle convolution (Mid-level features)
model.features[14].register_forward_hook(get_activation('layer_2_mid_level'))
# Layer 28: Deep convolution (High-level features)
model.features[28].register_forward_hook(get_activation('layer_3_high_level'))

# 4. Pass the image through the model
output = model(input_tensor)

## Step 5: Visualizing the Hierarchy

Now we inspect what the network "sees" at different depths. We define a helper function, visualize_layer_features, which plots the first 6 filters (feature maps) from our captured layers.

- Low-Level (Layer 1): You will see simple lines and edges, similar to our manual edge detector.

- Mid-Level (Layer 14): The patterns become more complex, capturing textures like fur or curves like eyes.

- High-Level (Layer 28): The features become abstract "blobs," representing high-level concepts (e.g., "ear detected") rather than visual details.

This visualization proves the Hierarchical Nature of Deep Learning: stacking simple layers creates complex understanding.

In [ ]:
def visualize_layer_features(layer_name, num_filters=6):
    activations = feature_maps[layer_name].squeeze().cpu()

    fig, axes = plt.subplots(1, num_filters, figsize=(20, 4))
    fig.suptitle(f"Features from: {layer_name}", fontsize=16)

    for i in range(num_filters):
        # Determine the 'best' filters to show (those with highest activation variance)
        # This helps avoid showing blank black squares
        img_data = activations[i].numpy()

        axes[i].imshow(img_data, cmap='viridis')
        axes[i].axis('off')
        axes[i].set_title(f"Filter {i}")

    plt.show()

print("Analyzing Hierarchical Feature Extraction...")

# 1. Low Level (Edges, Colors, Gradients)
visualize_layer_features('layer_1_low_level')
print("^^ Layer 1: Notice simple lines, diagonal edges, and color blobs. Similar to our manual Sobel filter.")

# 2. Mid Level (Textures, Curves)
visualize_layer_features('layer_2_mid_level')
print("^^ Layer 2: The patterns become more complex. You might see eye shapes, fur textures, or ear curves.")

# 3. High Level (Object Parts)
visualize_layer_features('layer_3_high_level')
print("^^ Layer 3: These are highly abstract. A single bright spot here might represent 'Cat Ear present' or 'Whiskers present'.")

## Step 6: The Full Pipeline Visualization (Input → Output)

Now we put everything together to visualize the entire journey of the image through the VGG16 network. In this block, we attach hooks to every Convolutional and Pooling layer, not just a few.

The code captures detailed metadata (Kernel size, Stride, Filter count) and dynamically tracks the image resolution. It highlights Pooling Layers with red text whenever the image size shrinks (Downsampling, e.g., 224x224 → 112x112), helping you see exactly where the spatial compression happens. Finally, it displays the model's top 5 classification predictions, showing how these abstract feature maps are converted into a final decision (e.g., "Tabby Cat").

In [ ]:
layer_outputs = {}
layer_info = {} # Store text details (Kernel size, Filters, etc.)

def get_hook(name):
    def hook(model, input, output):
        layer_outputs[name] = output.detach()
    return hook

# Loop through model.features to capture sequence and metadata
print("Hooking layers and extracting architecture details...")
layer_num = 0
for i, layer in enumerate(model.features):
    name = ""
    details = ""

    if isinstance(layer, nn.Conv2d):
        name = f"{layer_num}_Conv2d"
        # Extract Conv Details
        details = f"Filters: {layer.out_channels} | Kernel: {layer.kernel_size} | Stride: {layer.stride}"
        layer.register_forward_hook(get_hook(name))
        layer_info[name] = details
        layer_num += 1

    elif isinstance(layer, nn.MaxPool2d):
        name = f"{layer_num}_MaxPool"
        # Extract Pool Details
        details = f"Pool Size: {layer.kernel_size} | Stride: {layer.stride}"
        layer.register_forward_hook(get_hook(name))
        layer_info[name] = details
        layer_num += 1

# --- 3. FORWARD PASS ---
with torch.no_grad():
    output = model(input_tensor)

# --- 4. VISUALIZATION ---
print("\n--- Visualizing the Pipeline (Input -> Conv -> Pool) ---")

# A. Show Input Image First
plt.figure(figsize=(5, 5))
plt.imshow(original_image)
plt.title(f"INPUT IMAGE\nShape: {original_image.size} | Channels: 3 (RGB)", fontsize=14, fontweight='bold', color='blue')
plt.axis('off')
plt.show()

# B. Helper function
def normalize_for_plot(f_img):
    return (f_img - f_img.min()) / (f_img.max() - f_img.min())

# C. Loop through captured layers
prev_shape = input_tensor.shape[2] # Start at 224

for layer_name, feature_map in layer_outputs.items():
    feature_map = feature_map.squeeze().cpu()
    current_shape = feature_map.shape[1] # Height
    current_filters = feature_map.shape[0] # Channels/Filters

    # 1. Check Resolution Drop
    res_text = f"{current_shape}x{current_shape}"
    if current_shape < prev_shape:
        res_text += " (⬇ DOWNSAMPLED)"
        title_color = 'red'
    else:
        title_color = 'black'
    prev_shape = current_shape

    # 2. Get Metadata
    meta = layer_info.get(layer_name, "")

    # Setup Plot
    num_filters = 6
    fig, axes = plt.subplots(1, num_filters, figsize=(20, 3.5))

    # Dynamic Title with ALL info
    # Format: Layer Name | Resolution | Hardware Details (Filters/Kernels)
    full_title = f"{layer_name}\nRes: {res_text} | {meta}"

    fig.suptitle(full_title, fontsize=13, y=1.05, color=title_color, fontweight='bold')

    for i in range(num_filters):
        if i < feature_map.shape[0]:
            ax = axes[i]
            img_data = normalize_for_plot(feature_map[i])
            ax.imshow(img_data, cmap='viridis')
            ax.axis('off')
            # Optional: Label filter number
            ax.set_title(f"Filter {i+1}", fontsize=9)

    plt.show()

# 6. FINAL OUTPUT: The Classification
# The feature maps are flattened and sent to the classifier (Dense Layers)
probabilities = torch.nn.functional.softmax(output[0], dim=0)

# Get ImageNet class labels
# (Downloading labels for readability)
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(labels_url).text.splitlines()

# Get Top 5 Predictions
top5_prob, top5_catid = torch.topk(probabilities, 5)

print("\n--- FINAL OUTPUT (The Decision) ---")
plt.figure(figsize=(10, 5))
bars = plt.bar(range(5), top5_prob.cpu().numpy(), align='center', alpha=0.7, color='green')
plt.xticks(range(5), [labels[i] for i in top5_catid], rotation=15, fontsize=12)
plt.ylabel('Confidence Score')
plt.title('What the AI thinks this is:')
plt.ylim(0, 1.1)

# Add percentage labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height, f'{height:.1%}', ha='center', va='bottom')

plt.show()

## Step 7: Parameter Analysis & Transfer Learning

Finally, we analyze the "weight" of the model. This block calculates the total number of learnable parameters (weights and biases) in VGG16, giving us a sense of its complexity (approx. 138 million parameters).

We then simulate a Transfer Learning scenario by "freezing" the feature extraction layers (setting requires_grad = False). This demonstrates how fine-tuning works: we keep the pre-learned visual features (edges, shapes) and only retrain the classifier at the end, drastically reducing the computational cost (by ~89% in this case).

In [ ]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

print("\n" + "="*50)
print("     MODEL ARCHITECTURE & TRAINING ANALYSIS     ")
print("="*50)

# 1. Full Model Stats
total, trainable = count_parameters(model)
print(f"\n[Scenario 1] Training from Scratch (Full VGG16)")
print(f"Total Parameters:      {total:,}")
print(f"Trainable Parameters:  {trainable:,}")
print(f"Model Size (approx):   {total * 4 / (1024**2):.2f} MB (assuming float32)")

# 2. Simulate Fine-Tuning (Freezing Feature Extractor)
# In Transfer Learning, we often freeze the 'features' and only train the 'classifier'
print("\n[Scenario 2] Fine-Tuning (Transfer Learning)")
print("Freezing feature extraction layers...")

# Freeze weights
for param in model.features.parameters():
    param.requires_grad = False

total_ft, trainable_ft = count_parameters(model)

print(f"Total Parameters:      {total_ft:,}")
print(f"Trainable Parameters:  {trainable_ft:,} (Only Classifier layers)")
print(f"Reduction in Compute:  {(1 - trainable_ft/total_ft)*100:.1f}% less parameters to update")

print("\n" + "="*50)

## Visualizing All Filters (The "Grid" View)

In this advanced visualization step, we move beyond just looking at a few examples. We want to see the complete set of features—all 64, 128, 256, or 512 activation maps—at every single layer.

To handle this massive amount of data without crashing the notebook, we use a "Canvas Stitching" technique:

- make_grid: We stitch hundreds of small filter images into one large, high-resolution "poster" for each layer.

- Normalization: We normalize each filter individually so even faint activations are visible.

- Archiving: Since these images are large, the code automatically saves them to a folder and zips them up (vgg16_layer_visualizations.zip). This allows you to download the full "brain scan" of the AI and inspect the tiny 7x7 filters offline.

In [ ]:
from torchvision import utils
import os
import shutil

output_dir = "vgg16_visualizations"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir)

layer_outputs = {}
layer_info = {}

def get_hook(name):
    def hook(model, input, output):
        layer_outputs[name] = output.detach()
    return hook

layer_num = 0
for i, layer in enumerate(model.features):
    name = ""
    details = ""
    if isinstance(layer, nn.Conv2d):
        name = f"{layer_num:02d}_Conv2d" # Added padding 01, 02 for sorting
        details = f"Filters: {layer.out_channels} | Kernel: {layer.kernel_size}"
        layer.register_forward_hook(get_hook(name))
        layer_info[name] = details
        layer_num += 1
    elif isinstance(layer, nn.MaxPool2d):
        name = f"{layer_num:02d}_MaxPool"
        details = f"Pool: {layer.kernel_size}"
        layer.register_forward_hook(get_hook(name))
        layer_info[name] = details
        layer_num += 1

# --- 3. FORWARD PASS ---
with torch.no_grad():
    output = model(input_tensor)

# --- 4. ADVANCED VISUALIZATION (ALL FILTERS) ---
print(f"Processing layers... Images will be saved to '/{output_dir}' folder.\n")

prev_shape = input_tensor.shape[2]

for layer_name, feature_map in layer_outputs.items():
    feature_map = feature_map.squeeze().cpu() # Shape: [Filters, Height, Width]

    current_shape = feature_map.shape[1]
    num_filters = feature_map.shape[0]

    # 1. Info Text
    res_text = f"{current_shape}x{current_shape}"
    if current_shape < prev_shape:
        res_text += " (⬇ DOWNSAMPLED)"
    prev_shape = current_shape
    meta = layer_info.get(layer_name, "")

    print(f"Generating grid for {layer_name}: {num_filters} filters...")

    # 2. CREATE THE GRID (The "Canvas Stitching" Trick)
    # This function takes all [N, H, W] images and tiles them into one big image
    # We map single channel (grayscale) to 3 channels so we can apply colormap later if needed
    # padding=2 puts a 2px black line between filters so you can distinguish them

    # We normalize each filter individually for better visibility
    # (Otherwise weak filters look pitch black)
    f_map_norm = feature_map.clone()
    for f in range(num_filters):
        f_min, f_max = f_map_norm[f].min(), f_map_norm[f].max()
        if f_max > f_min: # Avoid division by zero
            f_map_norm[f] = (f_map_norm[f] - f_min) / (f_max - f_min)

    # Make Grid: automatically arranges 64 filters into 8x8, or 512 into roughly 22x23
    # nrow=int(np.sqrt(num_filters)) makes it roughly square
    grid_img_tensor = utils.make_grid(f_map_norm.unsqueeze(1), nrow=int(np.sqrt(num_filters)), padding=1, normalize=False, pad_value=0.5)

    # Convert to Numpy for Plotting: [C, H, W] -> [H, W, C]
    grid_img_np = grid_img_tensor.permute(1, 2, 0).numpy()

    # 3. PLOT INLINE (High Res)
    plt.figure(figsize=(10, 10)) # Keep figure size constant, resolution handles the detail
    plt.imshow(grid_img_np, cmap='viridis') # viridis adds the heat color

    title = f"{layer_name} | {res_text} | {meta}"
    plt.title(title, fontsize=12, fontweight='bold')
    plt.axis('off')

    # 4. SAVE TO DISK
    # Save using Matplotlib to keep the title and colormap
    save_path = os.path.join(output_dir, f"{layer_name}.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight') # dpi=150 gives high quality
    plt.show()
    plt.close() # Close memory to prevent crash

# --- 5. ZIP AND DOWNLOAD ---
print("\n" + "="*50)
print("     ZIPPING IMAGES FOR DOWNLOAD     ")
print("="*50)

shutil.make_archive('vgg16_layer_visualizations', 'zip', output_dir)
print(f"All layer grids saved to 'vgg16_layer_visualizations.zip'. Check your file browser!")

## Full Feature Grid & Attention Heatmap

This final block performs the heavy lifting of our analysis. It does two critical things:

- Grid Visualization (The "Brain Scan"): Instead of showing just a few filters, it takes every single filter from every layer (up to 512 at the end), stitches them into a massive high-resolution grid, and saves them to your local folder. This prevents the notebook from crashing while preserving the full detail for you to inspect later.

- The Master Heatmap (The "Aha!" Moment): It takes the final, abstract 7×7 output—which looks like random noise to humans—and averages it out. This reveals the "Attention Map," a heatmap showing exactly which parts of the image the AI found most interesting. By overlaying this on the original photo, we can prove the model is focusing on the cat's face and ignoring the background.

In [ ]:
layer_outputs = {}
layer_info = {}

def get_hook(name):
    def hook(model, input, output):
        layer_outputs[name] = output.detach()
    return hook

print("Attaching Hooks...")
layer_num = 0
for i, layer in enumerate(model.features):
    name = ""
    details = ""
    # We format with :02d (e.g., 01, 02) to keep sorting correct
    if isinstance(layer, nn.Conv2d):
        name = f"{layer_num:02d}_Conv2d"
        details = f"Filters: {layer.out_channels} | Kernel: {layer.kernel_size}"
        layer.register_forward_hook(get_hook(name))
        layer_info[name] = details
        layer_num += 1
    elif isinstance(layer, nn.MaxPool2d):
        name = f"{layer_num:02d}_MaxPool"
        details = f"Pool: {layer.kernel_size}"
        layer.register_forward_hook(get_hook(name))
        layer_info[name] = details
        layer_num += 1

# --- 3. FORWARD PASS ---
with torch.no_grad():
    output = model(input_tensor)

# --- 4. VISUALIZATION (GRID VIEW) ---
print(f"Processing layers... Images will be saved to '/{output_dir}' folder.\n")

prev_shape = input_tensor.shape[2]
last_layer_name = "" # Keep track of the last layer for the heatmap step later
last_feature_map = None

for layer_name, feature_map in sorted(layer_outputs.items()):
    feature_map = feature_map.squeeze().cpu()

    # Store for Section 5
    last_layer_name = layer_name
    last_feature_map = feature_map

    current_shape = feature_map.shape[1]
    num_filters = feature_map.shape[0]

    # Info Text
    res_text = f"{current_shape}x{current_shape}"
    if current_shape < prev_shape:
        res_text += " (⬇ DOWNSAMPLED)"
    prev_shape = current_shape
    meta = layer_info.get(layer_name, "")

    print(f"Generating grid for {layer_name}: {num_filters} filters... ({res_text})")

    # Normalize for Grid
    f_map_norm = feature_map.clone()
    for f in range(num_filters):
        f_min, f_max = f_map_norm[f].min(), f_map_norm[f].max()
        if f_max > f_min:
            f_map_norm[f] = (f_map_norm[f] - f_min) / (f_max - f_min)

    # Make Grid
    grid_img_tensor = utils.make_grid(f_map_norm.unsqueeze(1), nrow=int(np.sqrt(num_filters)), padding=1, normalize=False, pad_value=0.5)
    grid_img_np = grid_img_tensor.permute(1, 2, 0).numpy()

    # Save to Disk Only (To keep notebook clean, we won't plot all intermediate steps inline)
    plt.figure(figsize=(10, 10))
    plt.imshow(grid_img_np, cmap='viridis')
    plt.title(f"{layer_name} | {res_text} | {meta}", fontsize=12, fontweight='bold')
    plt.axis('off')
    save_path = os.path.join(output_dir, f"{layer_name}.png")
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.close()

# --- 5. THE "MASTER" HEATMAP (Solving the Noise Issue) ---
print("\n" + "="*50)
print("     CREATING AGGREGATE ATTENTION MAP     ")
print("="*50)

# We grab the last layer's feature map (which we saved in the loop above)
# This usually has shape [512, 7, 7]
print(f"Taking the final layer ({last_layer_name}) which has shape {last_feature_map.shape}...")

# Step A: Average all 512 filters into ONE 2D map
# This tells us "Where is the average activation highest?"
mean_heatmap = torch.mean(last_feature_map, dim=0).numpy()

# Step B: Normalize (0 to 1) so it can be viewed as an image
mean_heatmap = np.maximum(mean_heatmap, 0)
mean_heatmap /= np.max(mean_heatmap)

# Step C: Resize to match original image (Upsampling 7x7 -> 224x224)
heatmap_resized = cv2.resize(mean_heatmap, (224, 224))

# Step D: Convert to RGB Heatmap
# 1. Scale to 0-255 integers
heatmap_uint8 = np.uint8(255 * heatmap_resized)
# 2. Apply the Color Map (This turns grayscale into Blue-Red heat colors)
heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
# 3. Fix Color Space (OpenCV uses BGR, Matplotlib uses RGB)
# *** THIS WAS THE FIX ***
heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

# Step E: Overlay on Original Image
# Resize original image to match heatmap dimensions
original_resized = original_image.resize((224, 224))
original_np = np.array(original_resized)

# Blend: 60% Original + 40% Heatmap
# We need to ensure types match for blending
superimposed_img = cv2.addWeighted(original_np, 0.6, heatmap_colored, 0.4, 0)

# --- PLOT THE RESULT ---
fig, ax = plt.subplots(1, 3, figsize=(20, 8))

# 1. The "Noise" (One of the 512 filters)
# We show the 7x7 raw filter
ax[0].imshow(last_feature_map[0], cmap='viridis')
ax[0].set_title(f"Single Filter (Raw Output)\nShape: {last_feature_map.shape[1]}x{last_feature_map.shape[2]}", fontsize=14)
ax[0].axis('off')

# 2. The Aggregated Heatmap
ax[1].imshow(heatmap_resized, cmap='jet')
ax[1].set_title("Averaged Heatmap\n(Where the AI is looking)", fontsize=14)
ax[1].axis('off')

# 3. The Final Understanding
ax[2].imshow(superimposed_img)
ax[2].set_title("Overlay on Image\n(Contextual Focus)", fontsize=14)
ax[2].axis('off')

plt.show()